# github-dkg — Ingest GitHub knowledge into OriginTrail DKG v10

[![PyPI](https://img.shields.io/badge/pip-github--dkg-blue)](https://pypi.org/project/github-dkg/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)
[![Bounty](https://img.shields.io/badge/bounty-cfi--dkgv10--r1-orange)](https://docs.origintrail.io)

**Capture engineering knowledge that lives in GitHub — issues, PRs, reviews — as cryptographically-linked Knowledge Assets in DKG v10 Working Memory.**

This notebook runs against an **offline mock** of both GitHub and the DKG v10 node, so you can explore the package without API tokens or a running node. To run against real services, scroll to the *Live Mode* section at the bottom.

---

## What this package does

| Class | What it does |
|---|---|
| `GitHubClient` | Async wrapper over GitHub REST v3 with rate-limit detection |
| `DKGClient` | Async wrapper over DKG v10 HTTP API (port 9200) |
| `GitHubDKGIngestor` | Orchestrator: fetch → format as Markdown → write as Knowledge Asset |

## Pipeline

```
  GitHub repo → list issues / list PRs (paginated, rate-limit-aware)
                       │
                       ▼
                 format_issue / format_pull_request   (Markdown)
                       │
                       ▼
                  POST /api/memory/turn               (DKG v10)
                       │
                       ▼
          Knowledge Asset (UAL, structural triples, embeddings)
```

## Setup

Install `github-dkg` from PyPI.

In [ ]:
!pip install github-dkg -q

from github_dkg import DKGClient, GitHubClient, GitHubDKGIngestor
from github_dkg.formatter import format_issue, format_pull_request
print('github-dkg ready.')

## Offline mocks — realistic GitHub + DKG responses

`MockGitHubClient` returns canned issues, PRs, and review comments shaped like the real GitHub REST v3 API. `MockDKGClient` returns realistic DKG v10 turn responses (UAL, file hash, triple counts) and stores them so `memory_search` reflects what we actually ingested.

Both are drop-in replacements for the real clients.

In [ ]:
import hashlib, uuid

_FAKE_ISSUES = [
    {
        'number': 1, 'title': 'Add SSO login', 'state': 'closed',
        'state_reason': 'completed',
        'user': {'login': 'alice'}, 'labels': [{'name': 'feature'}],
        'created_at': '2025-11-04T09:00:00Z', 'closed_at': '2025-11-12T16:00:00Z',
        'updated_at': '2025-11-12T16:00:00Z',
        'body': 'Users want to log in with Google or GitHub SSO.',
        'html_url': 'https://github.com/example-org/repo/issues/1',
    },
    {
        'number': 2, 'title': 'Auth token leaks in error logs', 'state': 'open',
        'state_reason': None,
        'user': {'login': 'bob'}, 'labels': [{'name': 'bug'}, {'name': 'security'}],
        'created_at': '2026-01-22T10:00:00Z', 'closed_at': None,
        'updated_at': '2026-04-15T08:30:00Z',
        'body': 'Bearer tokens appear in 5xx error logs. We need to scrub them.',
        'html_url': 'https://github.com/example-org/repo/issues/2',
    },
]

_FAKE_PULLS = [
    {
        'number': 3, 'title': 'Implement Google SSO',
        'state': 'closed', 'draft': False,
        'user': {'login': 'alice'}, 'labels': [{'name': 'feature'}],
        'created_at': '2025-11-08T12:00:00Z',
        'merged_at': '2025-11-12T16:00:00Z',
        'closed_at': '2025-11-12T16:00:00Z',
        'updated_at': '2025-11-12T16:00:00Z',
        'body': 'Closes #1. Adds OAuth2 flow with state-token verification.',
        'html_url': 'https://github.com/example-org/repo/pull/3',
        'base': {'ref': 'main'}, 'head': {'ref': 'feat/google-sso'},
        'requested_reviewers': [{'login': 'carol'}],
    },
    {
        'number': 4, 'title': 'Scrub bearer tokens before logging',
        'state': 'open', 'draft': False,
        'user': {'login': 'bob'}, 'labels': [{'name': 'security'}],
        'created_at': '2026-04-12T11:00:00Z',
        'merged_at': None, 'closed_at': None,
        'updated_at': '2026-04-30T08:30:00Z',
        'body': 'Replaces bearer tokens with `[REDACTED]` in the structured log middleware. Refs #2.',
        'html_url': 'https://github.com/example-org/repo/pull/4',
        'base': {'ref': 'main'}, 'head': {'ref': 'fix/log-scrub'},
        'requested_reviewers': [],
    },
]

_FAKE_REVIEWS = {
    3: [
        {'user': {'login': 'carol'}, 'state': 'APPROVED',
         'submitted_at': '2025-11-12T15:30:00Z',
         'body': 'Looks good. State-token check is solid.'},
    ],
    4: [
        {'user': {'login': 'carol'}, 'state': 'CHANGES_REQUESTED',
         'submitted_at': '2026-04-22T09:00:00Z',
         'body': 'Need to also scrub the cookie header, not just Authorization.'},
    ],
}
_FAKE_INLINE = {
    4: [
        {'user': {'login': 'carol'},
         'path': 'src/middleware/logging.py',
         'body': 'Cookie header values can also leak — please scrub here too.'},
    ],
}
_FAKE_COMMENTS = {
    1: [
        {'user': {'login': 'bob'}, 'created_at': '2025-11-05T08:00:00Z',
         'body': 'GitHub SSO would be nice for the OSS contributors.'},
    ],
    2: [
        {'user': {'login': 'alice'}, 'created_at': '2026-04-12T11:00:00Z',
         'body': 'Confirmed in production logs from last Friday.'},
    ],
}


class MockGitHubClient:
    """Drop-in for GitHubClient — returns canned issues/PRs/reviews offline."""

    def __init__(self, *_, **__): pass

    async def list_issues(self, owner, repo, state='all', since=None):
        for it in _FAKE_ISSUES:
            if since is None or it['updated_at'] >= since:
                yield it

    async def list_pulls(self, owner, repo, state='all', since=None):
        # Mimic the real /pulls endpoint with sort=updated&direction=desc
        for pr in sorted(_FAKE_PULLS, key=lambda p: p['updated_at'], reverse=True):
            if since and pr['updated_at'] < since:
                return
            yield pr

    async def get_issue(self, owner, repo, n):
        return next(i for i in _FAKE_ISSUES if i['number'] == n)

    async def get_pull(self, owner, repo, n):
        return next(p for p in _FAKE_PULLS if p['number'] == n)

    async def list_issue_comments(self, owner, repo, n): return _FAKE_COMMENTS.get(n, [])
    async def list_pull_reviews(self, owner, repo, n):  return _FAKE_REVIEWS.get(n, [])
    async def list_pull_comments(self, owner, repo, n): return _FAKE_INLINE.get(n, [])


class MockDKGClient:
    """Drop-in for DKGClient — returns realistic DKG v10 turn responses."""

    _CONTRACT = '0x5cAC41237127F94C2D21dAE0B14BFeFa3BDcAAa'

    def __init__(self, *_, **__):
        self._counter = 0
        self._store: list[dict] = []

    def _next_ual(self):
        self._counter += 1
        return f'did:dkg:otp:2043/{self._CONTRACT}/{self._counter:010d}'

    async def ping(self): return True

    async def memory_turn(self, context_graph_id, markdown,
                          session_uri=None, layer='wm', sub_graph_name=None):
        ual = self._next_ual()
        n = self._counter
        record = {
            'entityUri': ual,
            'label': markdown.splitlines()[0][:80],
            'snippet': markdown[:240],
            'similarity': round(max(0.70, 0.97 - n * 0.02), 2),
            'memoryLayer': layer,
            'sourceFile': f'item_{n:04d}.md',
        }
        self._store.append(record)
        return {
            'turnUri': ual,
            'fileHash': hashlib.sha256(markdown.encode()).hexdigest(),
            'layer': layer,
            'graph': f'urn:graph:{context_graph_id}',
            'structuralTripleCount': 8 + (n % 6),
            'semanticTripleCount': 5 + (n % 4),
            'totalQuads': 13 + (n % 10),
            'embeddingId': f'emb-{uuid.uuid4().hex[:16]}',
            'sessionUri': session_uri,
        }

    async def memory_search(self, context_graph_id, query, limit=20, memory_layers=None):
        q = query.lower()
        ranked = sorted(
            [r for r in self._store if q in r['snippet'].lower() or q in r['label'].lower()],
            key=lambda r: -r['similarity'],
        )[:limit]
        return {'query': query, 'contextGraphId': context_graph_id,
                'resultCount': len(ranked), 'results': ranked}

    async def assertion_promote(self, name, context_graph_id, entities=None):
        return {'status': 'shared', 'name': name, 'sharedUri': f'dkg://swm/{name}'}


mock_gh = MockGitHubClient()
mock_dkg = MockDKGClient()
print('Mocks ready — MockGitHubClient + MockDKGClient.')

---
## Demo 1 — Single-item ingest

Ingest one issue. The package fetches the GitHub item, formats it as Markdown with author/state/labels/comments, and stores it as a Knowledge Asset on the DKG.

Each call to `ingestor.ingest_issue()` performs **one** `POST /api/memory/turn` and returns the **UAL** (Universal Asset Locator) of the resulting Knowledge Asset.

In [ ]:
ingestor = GitHubDKGIngestor(
    dkg=mock_dkg,
    github=mock_gh,
    context_graph_id='demo-cg',
    layer='wm',
)

# Inspect the Markdown that the package generates before sending
issue = await mock_gh.get_issue('example-org', 'repo', 2)
comments = await mock_gh.list_issue_comments('example-org', 'repo', 2)
preview = format_issue(issue, comments, 'example-org', 'repo')
print('--- formatted Markdown sent to /api/memory/turn ---')
print(preview)
print()

resp = await ingestor.ingest_issue('example-org', 'repo', 2)
print(f"Ingested issue #2 → UAL: {resp['turnUri']}")
print(f"  totalQuads={resp['totalQuads']}  fileHash={resp['fileHash'][:16]}...")

---
## Demo 2 — Bulk ingest with `since=`

`ingest_repo()` streams every issue and PR through `asyncio.Semaphore`-bounded concurrent writes. The `since=` cutoff is honoured for both endpoints — for PRs the package uses `sort=updated&direction=desc` and stops when results fall below the cutoff (the `/pulls` endpoint has no native `since` filter).

In [ ]:
result = await ingestor.ingest_repo(
    owner='example-org',
    repo='repo',
    since='2026-01-01T00:00:00Z',
)

print(f'issues_ingested  : {result.issues_ingested}')
print(f'pulls_ingested   : {result.pulls_ingested}')
print(f'errors           : {len(result.errors)}')
print(f'turn_uris        :')
for u in result.turn_uris:
    print(f'   {u}')

---
## Demo 3 — Search the ingested knowledge

Knowledge Assets are queryable via the DKG node's tri-modal search (vector + SPARQL + text). `MockDKGClient.memory_search` does a simple substring match here; the real node returns ranked results by semantic similarity.

An agent could then plug this into LangChain via [`langchain-dkg`](https://pypi.org/project/langchain-dkg/) — the GitHub knowledge becomes its memory.

In [ ]:
for q in ['token', 'SSO', 'review']:
    hits = await mock_dkg.memory_search('demo-cg', q, limit=3)
    print(f'\n  query : {q!r}   ({hits["resultCount"]} hits)')
    for item in hits['results']:
        print(f"    [{item['similarity']:.2f}] {item['label']}")
        print(f"           {item['entityUri']}")

---
## Demo 4 — Promote a key asset to Shared Working Memory

Working Memory is private to your node. Shared Working Memory is **gossip-replicated** across the paranet — useful for team-wide knowledge.

Promotion is **always explicit** and Curator-authorized. Architecture-decision PRs are a typical candidate (see `examples/workflow.yml` for an `if: contains(labels, 'architecture-decision')` gate).

In [ ]:
# Promote the most recent ingest to SWM
promote_uri = result.turn_uris[-1]
promoted = await ingestor.promote(promote_uri)
print(f'Promoted  : {promote_uri}')
print(f'Result    : {promoted}')

---
## Live mode

Replace `dkg=mock_dkg, github=mock_gh` with real clients to run against a live DKG v10 node and the real GitHub API.

```python
import os
from github_dkg import DKGClient, GitHubClient, GitHubDKGIngestor

dkg = DKGClient(token=os.environ['DKG_TOKEN'])         # default URL: localhost:9200
gh  = GitHubClient(token=os.environ['GITHUB_TOKEN'])

ingestor = GitHubDKGIngestor(
    dkg=dkg, github=gh,
    context_graph_id='your-cg-id',
    layer='wm',
    concurrency=5,
)

result = await ingestor.ingest_repo('OriginTrail', 'dkg-integrations')
```

Required env: `DKG_TOKEN` (from `dkg auth show`), `GITHUB_TOKEN` (a personal access token or `${{ github.token }}` inside an Action).

## Links

- **GitHub:** https://github.com/spangers11/github-dkg
- **PyPI:** `pip install github-dkg`
- **OriginTrail DKG v10 docs:** https://docs.origintrail.io
- **Bounty programme:** `cfi-dkgv10-r1`